# 🏦 Banking Loan & FAQ Agentic RAG Assistant
### Portfolio Project — Open Source LLM Stack

---

## 📖 What This Project Does

This project builds an **AI-powered Banking Assistant** that can:
- Answer **loan-related questions** (eligibility, EMI, interest rates)
- Answer **customer FAQ questions** (credit cards, accounts, policies)
- **Calculate loan EMI** automatically using a built-in calculator tool
- **Search the web** for latest RBI guidelines or banking news
- **Search internal documents** (your uploaded loan/FAQ PDFs)

---

## 🏗️ Project Architecture — How Everything Fits Together

```
┌─────────────────────────────────────────────────────┐
│                    USER QUESTION                     │
└─────────────────────┬───────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────┐
│                  AGENT (Top Layer)                   │
│   "Should I search docs, calculate, or web search?" │
└──────┬──────────────┬──────────────┬────────────────┘
       │              │              │
       ▼              ▼              ▼
┌────────────┐ ┌────────────┐ ┌────────────┐
│  RAG Tool  │ │ Calculator │ │ Web Search │
│(Your Docs) │ │ (EMI/Math) │ │(DuckDuckGo)│
└──────┬─────┘ └─────┬──────┘ └─────┬──────┘
       │             │              │
       └─────────────┴──────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────┐
│                LLM — Mistral-7B                      │
│  (Generates final answer using retrieved context)    │
└─────────────────────────────────────────────────────┘
```

---

## 🧠 Key Concepts Explained Simply

| Term | What It Means | Analogy |
|------|--------------|---------|
| **LLM** | AI model that understands and generates text | A very smart employee |
| **RAG** | Searching documents before answering | Employee reads the manual first |
| **Agent** | Decides which tool to use | Manager deciding which department to call |
| **FAISS** | Database that stores text as vectors for fast search | A very smart filing cabinet |
| **Embeddings** | Converting text into numbers so we can compare similarity | Translating words into coordinates on a map |
| **Chunking** | Splitting long documents into smaller pieces | Cutting a book into chapters |

---

## 📁 What We'll Build

```
banking-rag/
├── 📓 Banking_RAG_Assistant.ipynb   ← This notebook
├── 📄 requirements.txt              ← All Python packages needed
├── 📂 banking_docs/
│   ├── loan_policy.txt              ← Sample loan policy document
│   ├── faq.txt                      ← Customer FAQ document
│   └── credit_card_policy.txt       ← Credit card terms
└── 📂 faiss_index/                  ← Auto-generated vector database
```


---
## ⚙️ Phase 0 — Install Dependencies

### 📖 What & Why
We need several Python libraries:
- **langchain** — the main framework that connects LLM + tools + agent together
- **langchain-huggingface** — connects LangChain specifically to HuggingFace models
- **faiss-cpu** — the vector database where we store document embeddings
- **sentence-transformers** — converts text into numerical vectors (embeddings)
- **pypdf** — reads PDF files so we can ingest them
- **duckduckgo-search** — free web search, no API key needed
- **streamlit** — builds the web UI later

> ⚠️ **If running locally:** Restart the kernel after this cell runs once.


In [1]:
# Install all required packages
# The -q flag means "quiet" — reduces installation output noise
!pip install -q \
    langchain langchain-community langchain-huggingface langchainhub \
    faiss-cpu pypdf sentence-transformers \
    duckduckgo-search streamlit \
    python-dotenv pandas

print("✅ All packages installed successfully!")


✅ All packages installed successfully!


---
## 🔑 Phase 1 — Configuration & Settings

### 📖 What & Why
Before building anything, we define all settings in one place.
This is good practice — if you want to change the model or chunk size,
you only change it here, not scattered throughout the code.

### 🔐 HuggingFace Token
- HuggingFace hosts open-source models like Mistral-7B for free
- You need a free account + token to access the models
- Get yours at: https://huggingface.co/settings/tokens
- Read access is enough — you don't need write access

### 🤖 Model Choice
- **Mistral-7B-Instruct** — Great balance of quality and speed. Best choice.
- **Zephyr-7B** — Also good, slightly more conversational


In [32]:
import os

# ─────────────────────────────────────────────
# 🔐 YOUR HUGGINGFACE TOKEN
# ─────────────────────────────────────────────
# Paste your token here (get it from huggingface.co/settings/tokens)
HF_TOKEN = ""

# Set it as an environment variable so all libraries can access it
#os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

# ─────────────────────────────────────────────
# 🤖 MODEL SETTINGS
# ─────────────────────────────────────────────
# The LLM hosted on HuggingFace that will generate answers
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

# The model that converts text → numbers (embeddings) for similarity search
# all-MiniLM-L6-v2 is fast, lightweight, and works great for most use cases
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ─────────────────────────────────────────────
# 📄 CHUNKING SETTINGS
# ─────────────────────────────────────────────
# CHUNK_SIZE: How many characters per chunk when splitting documents
# Too large = less precise retrieval | Too small = loses context
CHUNK_SIZE = 500

# CHUNK_OVERLAP: How many characters overlap between consecutive chunks
# Overlap prevents losing information at chunk boundaries
CHUNK_OVERLAP = 50

# TOP_K: How many chunks to retrieve for each query
# More chunks = more context but slower and may confuse the LLM
TOP_K = 6

# ─────────────────────────────────────────────
# 🏦 BANKING DOMAIN SETTINGS
# ─────────────────────────────────────────────
# The folder where we'll store our banking documents
DOCS_FOLDER = "banking_docs"

# Where to save the FAISS vector index (so we don't re-embed every time)
INDEX_PATH = "faiss_index"

print("=" * 50)
print(f"LLM Model      : {LLM_MODEL}")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"Chunk Size     : {CHUNK_SIZE} chars  |  Overlap: {CHUNK_OVERLAP}")
print(f"Top-K Retrieval: {TOP_K} chunks")
print(f"Docs Folder    : {DOCS_FOLDER}/")
print("=" * 50)
print("✅ Configuration ready!")


LLM Model      : mistralai/Mistral-7B-Instruct-v0.2
Embedding Model: sentence-transformers/all-MiniLM-L6-v2
Chunk Size     : 500 chars  |  Overlap: 50
Top-K Retrieval: 6 chunks
Docs Folder    : banking_docs/
✅ Configuration ready!


---
## 📄 Phase 2 — Create Sample Banking Documents

### 📖 What & Why
In a real bank, you'd upload actual policy PDFs and FAQ documents.
Here we create realistic sample documents so you can run the project immediately.

These documents simulate:
1. **Loan Policy** — eligibility criteria, interest rates, EMI rules
2. **Customer FAQ** — common questions about accounts, cards, services
3. **Credit Card Policy** — credit limits, rewards, fees

### 💡 In Production
Replace these with real documents from your bank's knowledge base.
The system works with any PDF or TXT file — just drop them in the folder.


In [33]:
import os

# Create the folder to store banking documents
os.makedirs(DOCS_FOLDER, exist_ok=True)

# ─────────────────────────────────────────────
# 📄 Document 1: Loan Policy
# ─────────────────────────────────────────────
# This simulates a bank's internal loan policy document
loan_policy = """
NATIONAL BANK — PERSONAL LOAN & HOME LOAN POLICY DOCUMENT
==========================================================

1. PERSONAL LOAN ELIGIBILITY CRITERIA
--------------------------------------
- Minimum age: 21 years | Maximum age: 60 years at loan maturity
- Minimum monthly income: ₹25,000 (salaried) | ₹30,000 (self-employed)
- Minimum credit score (CIBIL): 700 and above
- Minimum employment tenure: 2 years (salaried) | 3 years (self-employed)
- Indian resident or NRI with valid documentation

2. PERSONAL LOAN DETAILS
--------------------------
- Loan amount: ₹50,000 to ₹40,00,000
- Interest rate: 10.5% to 18% per annum (based on credit profile)
- Loan tenure: 12 months to 60 months
- Processing fee: 1% to 2% of loan amount + GST
- Prepayment charges: 2% on outstanding principal after 6 EMIs
- No collateral required for personal loans

3. HOME LOAN ELIGIBILITY CRITERIA
-----------------------------------
- Minimum age: 23 years | Maximum age: 70 years at loan maturity
- Minimum monthly income: ₹40,000
- Minimum credit score (CIBIL): 750 and above
- Maximum LTV (Loan-to-Value): 80% of property value

4. HOME LOAN DETAILS
----------------------
- Loan amount: ₹5,00,000 to ₹5,00,00,000
- Interest rate: 8.5% to 10.5% per annum
- Loan tenure: 5 years to 30 years
- Processing fee: 0.5% of loan amount (max ₹15,000)
- Tax benefit: Up to ₹2,00,000 on interest (Section 24) and ₹1,50,000 on principal (Section 80C)

5. REQUIRED DOCUMENTS
-----------------------
For Salaried Employees:
  - PAN card and Aadhaar card
  - Last 3 months salary slips
  - Last 6 months bank statements
  - Form 16 or ITR for last 2 years
  - Employment letter or offer letter

For Self-Employed:
  - PAN card and Aadhaar card
  - ITR for last 3 years with CA certification
  - Last 12 months bank statements
  - Business registration proof
  - GST returns for last 1 year

6. EMI CALCULATION FORMULA
----------------------------
EMI = P × R × (1 + R)^N / ((1 + R)^N - 1)
Where:
  P = Principal loan amount
  R = Monthly interest rate (Annual Rate / 12 / 100)
  N = Number of monthly installments (tenure in months)

Example: ₹5,00,000 loan at 12% for 36 months
  R = 12/12/100 = 0.01
  EMI = 500000 × 0.01 × (1.01)^36 / ((1.01)^36 - 1)
  EMI = ₹16,607 per month

7. LOAN REJECTION REASONS
---------------------------
- CIBIL score below minimum threshold
- Insufficient income to support EMI (EMI should not exceed 50% of monthly income)
- Incomplete or fraudulent documentation
- Existing high debt obligations
- Unstable employment history
- Property legal issues (for home loans)

8. LOAN PROCESSING TIME
-------------------------
- Personal loan: 2-7 working days after document submission
- Home loan: 15-30 working days after document submission
- Instant personal loan (pre-approved customers): Same day disbursement
"""

with open(f"{DOCS_FOLDER}/loan_policy.txt", "w") as f:
    f.write(loan_policy)
print("✅ loan_policy.txt created")

# ─────────────────────────────────────────────
# 📄 Document 2: Customer FAQ
# ─────────────────────────────────────────────
faq_document = """
NATIONAL BANK — CUSTOMER FREQUENTLY ASKED QUESTIONS (FAQ)
==========================================================

SECTION A: SAVINGS ACCOUNT
-----------------------------

Q: What is the minimum balance required for a savings account?
A: Regular Savings Account requires ₹1,000 minimum balance. Premium Savings Account requires
   ₹10,000. Zero Balance accounts are available for students and Jan Dhan account holders.

Q: What is the interest rate on savings accounts?
A: Regular Savings Account: 3.5% per annum. Premium Savings Account: 4.0% per annum.
   Interest is calculated daily and credited quarterly.

Q: How many free ATM transactions do I get per month?
A: You get 5 free transactions at our own bank ATMs (unlimited at home branch).
   For other bank ATMs, you get 3 free transactions per month. After that,
   ₹20 per transaction is charged.

Q: How do I activate internet banking?
A: Visit your nearest branch with your Aadhaar and PAN card, or register online
   at our website using your registered mobile number and debit card details.
   An OTP will be sent to your registered mobile number for verification.

SECTION B: LOANS
-----------------

Q: How long does it take to get a personal loan approved?
A: For pre-approved customers, same-day disbursement is possible.
   For regular applications, it takes 2 to 7 working days after all documents are submitted.

Q: Can I repay my loan early?
A: Yes. Prepayment is allowed after 6 EMIs. Prepayment charge is 2% on the outstanding principal.
   For home loans, no prepayment charges apply if you pay from your own funds (not balance transfer).

Q: What happens if I miss an EMI payment?
A: A late payment fee of ₹500 or 2% of the EMI amount (whichever is higher) will be charged.
   Continuous default of 3 EMIs will be reported to CIBIL and may affect your credit score.
   The bank will also initiate recovery proceedings after 90 days of default (NPA classification).

Q: Can I get a loan if my CIBIL score is below 700?
A: Standard personal loans require a minimum CIBIL score of 700. However, secured loans
   (against FD, gold, or property) may be available with a lower credit score.
   We recommend improving your credit score before applying for unsecured loans.

SECTION C: CREDIT CARDS
------------------------

Q: How do I apply for a credit card?
A: You can apply online at our website, at any branch, or through our mobile app.
   Eligibility: minimum age 21 years, minimum annual income ₹3,00,000, CIBIL score 700+.

Q: What is the credit card billing cycle?
A: The billing cycle is 30 days. Your statement is generated on the statement date,
   and the due date is 20 days after the statement date.
   Paying the full amount by the due date avoids interest charges.

Q: What are the charges if I pay only the minimum amount due?
A: Interest is charged at 3.5% per month (42% per annum) on the outstanding balance
   from the date of transaction if you do not pay the full amount.

Q: How do I block my lost or stolen card?
A: Call our 24/7 helpline at 1800-XXX-XXXX immediately.
   You can also block it instantly through our mobile app under Card Settings.
   Report the loss to the police and submit an FIR copy to the bank within 24 hours.

SECTION D: FIXED DEPOSITS
---------------------------

Q: What are the current FD interest rates?
A: 7 days to 30 days: 3.50% | 31 days to 90 days: 4.50% | 91 days to 180 days: 5.50%
   181 days to 1 year: 6.50% | 1 year to 2 years: 7.25% | 2 years to 5 years: 7.50%
   Senior citizens get 0.50% extra on all tenures.

Q: Can I break my FD before maturity?
A: Yes, but a premature withdrawal penalty of 1% is deducted from the applicable interest rate.
   No penalty for FDs broken after 7 days from the date of deposit.

SECTION E: GENERAL
--------------------

Q: What are the bank's working hours?
A: Monday to Friday: 10:00 AM to 4:00 PM | Saturday: 10:00 AM to 1:00 PM
   Closed on Sundays and public holidays. ATMs are available 24/7.

Q: How do I update my mobile number or address?
A: Visit your home branch with your Aadhaar card and a self-attested copy.
   Alternatively, update via internet banking under Profile Settings (mobile number only).

Q: Is my deposit insured?
A: Yes. Deposits up to ₹5,00,000 per depositor per bank are insured by
   Deposit Insurance and Credit Guarantee Corporation (DICGC) under RBI regulations.
"""

with open(f"{DOCS_FOLDER}/customer_faq.txt", "w") as f:
    f.write(faq_document)
print("✅ customer_faq.txt created")

# ─────────────────────────────────────────────
# 📄 Document 3: Credit Card Policy
# ─────────────────────────────────────────────
credit_card_policy = """
NATIONAL BANK — CREDIT CARD POLICY & FEATURES
===============================================

1. CREDIT CARD TYPES & ELIGIBILITY
-------------------------------------

CLASSIC CARD:
  - Annual income required: ₹3,00,000
  - Credit limit: ₹25,000 to ₹1,00,000
  - Annual fee: ₹500 (waived if annual spend exceeds ₹50,000)
  - Rewards: 1 reward point per ₹100 spent

GOLD CARD:
  - Annual income required: ₹6,00,000
  - Credit limit: ₹1,00,000 to ₹5,00,000
  - Annual fee: ₹1,000 (waived if annual spend exceeds ₹1,50,000)
  - Rewards: 2 reward points per ₹100 spent
  - Lounge access: 2 complimentary domestic airport lounge visits per quarter

PLATINUM CARD:
  - Annual income required: ₹12,00,000
  - Credit limit: ₹5,00,000 to ₹20,00,000
  - Annual fee: ₹2,500 (waived if annual spend exceeds ₹3,00,000)
  - Rewards: 5 reward points per ₹100 spent
  - Lounge access: Unlimited domestic + 4 international per year
  - Concierge service, travel insurance up to ₹1 crore

2. INTEREST & CHARGES
-----------------------
  - Purchase interest rate: 3.5% per month (42% per annum)
  - Cash advance fee: 2.5% of amount withdrawn (minimum ₹500)
  - Cash advance interest: Charged from day of withdrawal at 3.5% per month
  - Late payment fee: ₹100 to ₹1,300 depending on outstanding balance
  - Over-limit fee: 2.5% of over-limit amount (minimum ₹500)
  - Foreign transaction fee: 3.5% of transaction amount

3. REWARD POINTS POLICY
-------------------------
  - 1 reward point = ₹0.25 value
  - Points valid for 3 years from date of earning
  - Redeem against statement credit, flights, hotels, or Amazon vouchers
  - Bonus 500 points on first transaction within 30 days of card issuance
  - 10X reward points on birthday month spending

4. CREDIT LIMIT ENHANCEMENT
-----------------------------
  - Request after 6 months of card usage
  - Good repayment history and income proof required
  - Enhancement of up to 30% can be done instantly online
  - Higher enhancements require income document re-verification

5. SECURITY FEATURES
----------------------
  - EMV chip and PIN for all transactions
  - Virtual card number for online transactions
  - Real-time SMS and email alerts for every transaction
  - Instant card block via mobile app
  - Zero liability on fraudulent transactions (reported within 3 days)
"""

with open(f"{DOCS_FOLDER}/credit_card_policy.txt", "w") as f:
    f.write(credit_card_policy)
print("✅ credit_card_policy.txt created")

print(f"\n📂 All banking documents created in '{DOCS_FOLDER}/' folder")
print("   - loan_policy.txt")
print("   - customer_faq.txt")
print("   - credit_card_policy.txt")


✅ loan_policy.txt created
✅ customer_faq.txt created
✅ credit_card_policy.txt created

📂 All banking documents created in 'banking_docs/' folder
   - loan_policy.txt
   - customer_faq.txt
   - credit_card_policy.txt


---
## 🔍 Phase 3 — RAG Pipeline (The Knowledge Base)

### 📖 What Is RAG?
RAG stands for **Retrieval-Augmented Generation**.

The problem with a plain LLM is it doesn't know your bank's specific policies.
RAG solves this by:
1. **Ingesting** your documents and storing them in a vector database
2. **Retrieving** the most relevant chunks when a question is asked
3. **Passing** those chunks as context to the LLM so it answers accurately

### 🔢 What Are Embeddings?
Embeddings convert text into numbers (vectors). Similar texts get similar numbers.
So "What is the EMI for home loan?" and "home loan monthly payment" will have
very similar vectors — meaning the search finds relevant content even if words don't exactly match.

### 🗄️ What Is FAISS?
FAISS (Facebook AI Similarity Search) is a library that stores these vectors
and very quickly finds the most similar ones to your query vector.
Think of it as a search engine that understands meaning, not just keywords.

### 📦 What Is Chunking?
Documents are split into smaller pieces (chunks) because:
- LLMs have a context window limit
- Smaller, focused chunks retrieve more precisely
- Overlapping chunks prevent losing information at boundaries


In [34]:
# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────
# logging: Python's built-in module for printing status messages
import logging
# Path: Makes file path handling easier across Windows/Mac/Linux
from pathlib import Path
# List, Dict, Any: Type hints — they make the code easier to understand
# and catch bugs early. List[str] means "a list of strings", etc.
from typing import List, Dict, Any

# RecursiveCharacterTextSplitter: Splits documents into chunks
# It tries to split on paragraphs first, then sentences, then words
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Document loaders — each handles a different file type
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# FAISS: Vector database for storing and searching embeddings
from langchain_community.vectorstores import FAISS

# HuggingFaceEmbeddings: Converts text → numerical vectors
# Uses sentence-transformers models hosted on HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings

# Document: LangChain's data structure — holds text content + metadata
# metadata = info about the document like filename, page number
from langchain_core.documents import Document

# Set logging to WARNING so we don't see too much output
logging.basicConfig(level=logging.WARNING)

In [35]:
# ─────────────────────────────────────────────────────────────────────
# RAG PIPELINE CLASS
# ─────────────────────────────────────────────────────────────────────
class BankingRAGPipeline:
    """
    A complete Retrieval-Augmented Generation pipeline for banking documents.

    HOW IT WORKS:
    1. Load documents (PDF or TXT files)
    2. Split them into overlapping chunks
    3. Convert each chunk into a vector (embedding)
    4. Store vectors in FAISS database
    5. For any query, find the most similar chunks and return them

    PARAMETERS:
    - embedding_model: Which sentence-transformer to use for embeddings
    - chunk_size: Max characters per chunk (500 is a good default)
    - chunk_overlap: Characters shared between consecutive chunks (prevents info loss)
    - top_k: How many chunks to return per query
    - index_path: Where to save/load the FAISS index on disk
    """

    def __init__(
        self,
        embedding_model: str = EMBEDDING_MODEL,
        chunk_size: int = CHUNK_SIZE,
        chunk_overlap: int = CHUNK_OVERLAP,
        top_k: int = TOP_K,
        index_path: str = INDEX_PATH,
    ):
        # Store settings as instance variables so other methods can access them
        self.top_k = top_k
        self.index_path = index_path
        self.vectorstore = None  # Will be populated after ingestion

        print(f"⏳ Loading embedding model: {embedding_model}")
        print("   (This downloads ~90MB on first run, then cached locally)")

        # HuggingFaceEmbeddings: loads the sentence-transformer model
        # model_kwargs={"device": "cpu"} — use CPU (change to "cuda" if you have GPU)
        # normalize_embeddings=True — makes all vectors unit length for consistent similarity scores
        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )

        # RecursiveCharacterTextSplitter: Splits text intelligently
        # It tries these separators IN ORDER until chunks are small enough:
        # 1. Double newline (paragraph break) — best natural boundary
        # 2. Single newline
        # 3. Period (sentence end)
        # 4. Space (word boundary)
        # 5. Empty string (character level — last resort)
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ".", " ", ""],
        )

        print("✅ BankingRAGPipeline initialized!")
        print(f"   Chunk size: {chunk_size} | Overlap: {chunk_overlap} | Top-K: {top_k}")

    # ──────────────────────────────────────────────────────────────────
    # PRIVATE METHOD: Load a single file
    # Private methods start with _ (convention = "internal use only")
    # ──────────────────────────────────────────────────────────────────
    def _load_file(self, fp: Path) -> List[Document]:
        """
        Load a single file and return a list of Document objects.

        WHY SEPARATE LOADERS?
        - PDF files have a binary format — PyPDFLoader handles page extraction
        - TXT/MD files are plain text — TextLoader reads them directly
        - Each loader returns a list of Document objects, one per page/section

        WHAT IS A Document?
        - doc.page_content = the actual text
        - doc.metadata = dict with info like {"source": "loan_policy.txt", "page": 0}
        """
        try:
            suffix = fp.suffix.lower()  # e.g., ".pdf", ".txt", ".md"

            if suffix == ".pdf":
                # PyPDFLoader: reads each page as a separate Document
                # Returns list of Documents, one per page
                return PyPDFLoader(str(fp)).load()

            elif suffix in (".txt", ".md"):
                # TextLoader: reads entire file as one Document
                # encoding="utf-8" handles special characters like ₹, é, etc.
                return TextLoader(str(fp), encoding="utf-8").load()

            else:
                print(f"   ⚠️ Unsupported file type: {fp.name} — skipping")
                return []

        except Exception as e:
            # If any error occurs, warn and return empty list (don't crash)
            print(f"   ⚠️ Could not load {fp.name}: {e}")
            return []

    # ──────────────────────────────────────────────────────────────────
    # PUBLIC METHOD: Ingest documents
    # ──────────────────────────────────────────────────────────────────
    def ingest(self, source: str) -> int:
        """
        Main ingestion method — loads, chunks, embeds, and indexes documents.

        STEP-BY-STEP PROCESS:
        1. Find all files in the given folder (or single file)
        2. Load each file using the appropriate loader
        3. Split all documents into chunks using the text splitter
        4. Embed each chunk using the sentence-transformer model
        5. Store all embeddings in the FAISS vector database

        RETURNS: Number of chunks created and indexed

        WHY CHUNKS INSTEAD OF WHOLE DOCUMENTS?
        - LLMs have token limits (can't read 50 pages at once)
        - Smaller chunks = more precise retrieval
        - We find ONLY the relevant paragraphs, not the whole document
        """
        print(f"\n📂 Ingesting documents from: {source}")
        path = Path(source)

        # Collect all files to process
        if path.is_dir():
            # rglob("*") finds ALL files recursively (including subdirectories)
            files = list(path.rglob("*"))
        else:
            files = [path]

        # Load all files
        raw_docs: List[Document] = []
        for fp in files:
            if fp.is_file():
                loaded = self._load_file(fp)
                if loaded:
                    print(f"   📄 Loaded: {fp.name} ({len(loaded)} pages/sections)")
                    raw_docs.extend(loaded)

        if not raw_docs:
            raise ValueError(f"No readable documents found at: {source}")

        # CHUNKING: Split all documents into smaller pieces
        # split_documents() handles a list of Document objects
        # It preserves metadata (source file, page number) in each chunk
        chunks = self.splitter.split_documents(raw_docs)
        print(f"\n✂️  Chunking: {len(raw_docs)} pages → {len(chunks)} chunks")

        # EMBEDDING + INDEXING: Convert chunks to vectors and store in FAISS
        # FAISS.from_documents() does TWO things at once:
        # 1. Calls self.embeddings to convert each chunk's text to a vector
        # 2. Stores those vectors in the FAISS index for fast similarity search
        print(f"⏳ Embedding {len(chunks)} chunks... (may take 1-2 minutes)")

        if self.vectorstore is None:
            # First time: create a brand new FAISS index from scratch
            self.vectorstore = FAISS.from_documents(chunks, self.embeddings)
        else:
            # Already have an index: add new chunks to existing index
            self.vectorstore.add_documents(chunks)

        print(f"✅ Successfully indexed {len(chunks)} chunks!")
        return len(chunks)

    # ──────────────────────────────────────────────────────────────────
    # PUBLIC METHOD: Retrieve relevant chunks
    # ──────────────────────────────────────────────────────────────────
    def retrieve(self, query: str) -> List[Dict[str, Any]]:
        """
        Find the most relevant document chunks for a given query.

        HOW IT WORKS:
        1. Convert the query text into a vector using the same embedding model
        2. FAISS searches the index for the most similar vectors (cosine similarity)
        3. Return the top-k chunks with their similarity scores

        WHAT IS SIMILARITY SCORE?
        - Score close to 0 = very similar (normalized vectors)
        - Score farther from 0 = less similar
        - Lower score = better match (it's a distance metric)

        RETURNS: List of dicts with keys: content, metadata, score
        """
        if self.vectorstore is None:
            raise RuntimeError("No documents ingested yet! Call .ingest() first.")

        # similarity_search_with_score returns (Document, score) tuples
        results = self.vectorstore.similarity_search_with_score(query, k=self.top_k)

        # Format results into clean dictionaries for easy use
        return [
            {
                "content": doc.page_content,      # The actual text chunk
                "metadata": doc.metadata,          # Source file, page number, etc.
                "score": float(score),             # Similarity score (lower = more similar)
            }
            for doc, score in results
        ]

    # ──────────────────────────────────────────────────────────────────
    # PUBLIC METHODS: Save and load the FAISS index
    # ──────────────────────────────────────────────────────────────────
    def save_index(self) -> None:
        """
        Save the FAISS index to disk so you don't re-embed documents every time.
        WHY? Embedding takes time. Save once, load instantly next time.
        """
        if self.vectorstore:
            self.vectorstore.save_local(self.index_path)
            print(f"💾 Index saved to '{self.index_path}/' folder")

    def load_index(self) -> None:
        """
        Load a previously saved FAISS index from disk.
        allow_dangerous_deserialization=True is required by FAISS security checks.
        """
        if Path(self.index_path).exists():
            self.vectorstore = FAISS.load_local(
                self.index_path,
                self.embeddings,
                allow_dangerous_deserialization=True,
            )
            print(f"📂 Index loaded from '{self.index_path}/'")
        else:
            print(f"⚠️ No saved index found at '{self.index_path}/'")

print("✅ BankingRAGPipeline class defined!")


✅ BankingRAGPipeline class defined!


### 3a — Ingest banking documents & test retrieval

In [36]:
# Create the RAG pipeline
rag = BankingRAGPipeline()

# Ingest all documents from our banking_docs folder
chunk_count = rag.ingest(DOCS_FOLDER)
print(f"\n🏦 Banking knowledge base ready with {chunk_count} chunks!")


⏳ Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
   (This downloads ~90MB on first run, then cached locally)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BankingRAGPipeline initialized!
   Chunk size: 500 | Overlap: 50 | Top-K: 6

📂 Ingesting documents from: banking_docs
   📄 Loaded: loan_policy.txt (1 pages/sections)
   📄 Loaded: customer_faq.txt (1 pages/sections)
   📄 Loaded: credit_card_policy.txt (1 pages/sections)

✂️  Chunking: 3 pages → 28 chunks
⏳ Embedding 28 chunks... (may take 1-2 minutes)
✅ Successfully indexed 28 chunks!

🏦 Banking knowledge base ready with 28 chunks!


In [37]:
# ─────────────────────────────────────────────
# TEST RETRIEVAL — See how the system finds relevant chunks
# ─────────────────────────────────────────────
test_queries = [
    "What is the minimum CIBIL score for personal loan?",
    "What are the credit card interest charges?",
    "How do I request a  credit card?",
]

for query in test_queries:
    print(f"\n🔍 Query: {query}")
    print("-" * 60)
    results = rag.retrieve(query)
    top = results[0]  # Show only the best match
    source = Path(top['metadata'].get('source', 'unknown')).name
    print(f"📄 Source: {source} | Similarity Score: {top['score']:.4f}")
    print(f"📝 Content preview: {top['content'][:500]}...")



🔍 Query: What is the minimum CIBIL score for personal loan?
------------------------------------------------------------
📄 Source: customer_faq.txt | Similarity Score: 0.4528
📝 Content preview: Q: Can I get a loan if my CIBIL score is below 700?
A: Standard personal loans require a minimum CIBIL score of 700. However, secured loans
   (against FD, gold, or property) may be available with a lower credit score.
   We recommend improving your credit score before applying for unsecured loans.

SECTION C: CREDIT CARDS
------------------------...

🔍 Query: What are the credit card interest charges?
------------------------------------------------------------
📄 Source: customer_faq.txt | Similarity Score: 0.7534
📝 Content preview: Q: What is the credit card billing cycle?
A: The billing cycle is 30 days. Your statement is generated on the statement date,
   and the due date is 20 days after the statement date.
   Paying the full amount by the due date avoids interest charges.

Q: What are th

---
## 🔧 Phase 4 — Banking Agent Tools

### 📖 What Are Tools?
Tools are functions the agent can call to get information or perform actions.
The agent reads the tool's description and decides intelligently which tool fits the question.

### 🏦 Our 3 Banking Tools

| Tool | When Agent Uses It | Example Question |
|------|-------------------|-----------------|
| `document_search` | Question about bank policies, eligibility, FAQs | "What documents do I need for a home loan?" |
| `loan_emi_calculator` | Any EMI or loan math calculation | "What will be my EMI for ₹5 lakh at 12% for 3 years?" |
| `web_search` | Current rates, RBI news, general banking info | "What is the current RBI repo rate?" |

### 💡 Why Tools Instead of Just the LLM?
- LLM might hallucinate wrong interest rates
- Calculator tools give 100% accurate math
- Web search gets real-time information the LLM wasn't trained on


In [38]:
# ─────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────
# 'tool' decorator: newer LangChain way to define tools
# The function name becomes the tool name
# The function docstring becomes the tool description (agent reads this!)
from langchain.tools import tool

# re: Python's built-in Regular Expression module
# Used to extract numbers from strings like "principal=500000, rate=12"
import re

# math: Python's built-in math module
# Gives us sqrt(), log(), sin(), cos(), pi, etc.
import math

# DuckDuckGoSearchRun: Free web search wrapper, no API key needed
from langchain_community.tools import DuckDuckGoSearchRun


# ═══════════════════════════════════════════════════════════════════════════
# TOOL 1: DOCUMENT SEARCH
# ═══════════════════════════════════════════════════════════════════════════
@tool
def document_search(query: str) -> str:
    """Search the bank's internal documents for information about loan policies,
    eligibility criteria, interest rates, required documents, credit card policies,
    customer FAQ, savings account rules, FD rates, and any banking procedures.
    Use this tool FIRST for any question about the bank's products or services.
    Input: a natural-language question or keywords about banking."""

    # WHY NO FACTORY FUNCTION ANYMORE?
    # With @tool decorator, the function lives in the notebook's global scope.
    # It can directly access 'rag' (our BankingRAGPipeline object) without
    # needing a closure/factory function. Simpler and cleaner.

    try:
        # Call RAG pipeline to retrieve the most relevant document chunks
        results = rag.retrieve(query)

        # If nothing found, tell the agent clearly
        if not results:
            return "No relevant information found in the banking documents."

        # Format each retrieved chunk so the LLM can read them clearly
        # enumerate(results, 1) starts counting from 1 instead of 0
        formatted_parts = []
        for i, result in enumerate(results, 1):
            # Path(...).name extracts just the filename, not the full path
            # e.g. "/home/user/banking_docs/loan_policy.txt" → "loan_policy.txt"
            source = Path(result['metadata'].get('source', 'unknown')).name

            # Build a clearly labelled block for each result
            formatted_parts.append(
                f"[Source {i}: {source}]\n{result['content']}"
            )

        # Join all chunks with blank lines between them for readability
        return "\n\n".join(formatted_parts)

    except RuntimeError as e:
        # RuntimeError is raised by RAGPipeline.retrieve() if nothing is ingested yet
        return f"Document search error: {str(e)}"


# ═══════════════════════════════════════════════════════════════════════════
# TOOL 2: LOAN EMI CALCULATOR
# ═══════════════════════════════════════════════════════════════════════════
@tool
def loan_emi_calculator(expression: str) -> str:
    """Calculate loan EMI (Equated Monthly Installment) or evaluate math.
    For EMI: use format 'EMI: principal=500000, rate=12, tenure=36'
      principal = loan amount in rupees
      rate      = annual interest rate in percent (e.g. 12 for 12%)
      tenure    = loan duration in months (e.g. 36 for 3 years)
    For general math: provide a Python expression e.g. '500000 * 0.01'
    Use this tool whenever the user asks for EMI, monthly payment, or loan math."""

    # ── WHY REGEX? ────────────────────────────────────────────────────────
    # The agent sends this tool a plain text string, not structured data.
    # Example input: "EMI: principal=500000, rate=12, tenure=36"
    #
    # We need to extract the three numbers (500000, 12, 36) from that string.
    # Regex lets us define a PATTERN and find matching text within a string.
    #
    # PATTERN BREAKDOWN:
    # (?i)          → case-insensitive (matches "EMI", "emi", "Emi")
    # emi           → literal text "emi"
    # .*            → any characters in between (flexible spacing/punctuation)
    # principal     → literal text "principal"
    # [=:]\s*       → followed by "=" or ":" then optional whitespace
    # ([\d.]+)      → CAPTURE GROUP — one or more digits or dots (the number)
    # .*            → any characters between fields
    # rate[=:]\s*([\d.]+)    → same pattern for rate
    # .*tenure[=:]\s*([\d.]+) → same pattern for tenure
    #
    # So for "EMI: principal=500000, rate=12, tenure=36":
    #   group(1) = "500000"
    #   group(2) = "12"
    #   group(3) = "36"
    # ─────────────────────────────────────────────────────────────────────

    emi_pattern = r"(?i)emi.*principal[=:]\s*([\d.]+).*rate[=:]\s*([\d.]+).*tenure[=:]\s*([\d.]+)"
    emi_match = re.search(emi_pattern, expression)

    if emi_match:
        # re.search() returns a Match object if pattern is found, else None
        # .group(1), .group(2), .group(3) retrieve the three captured numbers
        # float() converts the string "500000" → number 500000.0
        principal    = float(emi_match.group(1))   # P — loan amount
        annual_rate  = float(emi_match.group(2))   # Annual interest %
        tenure_months = float(emi_match.group(3))  # N — number of months

        # ── EMI FORMULA ───────────────────────────────────────────────
        # EMI = P × R × (1+R)^N / ((1+R)^N - 1)
        # Where R = monthly interest rate = annual_rate / 12 / 100
        # Example: 12% annual → 12/12/100 = 0.01 monthly (1% per month)
        r = annual_rate / 12 / 100   # Convert annual % to monthly decimal
        n = tenure_months

        if r == 0:
            # Edge case: 0% interest → just divide principal by months
            emi = principal / n
        else:
            # Standard EMI formula
            emi = principal * r * (1 + r)**n / ((1 + r)**n - 1)

        total_payment  = emi * n                     # Total amount paid over tenure
        total_interest = total_payment - principal   # How much extra you pay as interest

        # Return a nicely formatted breakdown
        return (
            f"💰 EMI Calculation:\n"
            f"   Loan Amount    : ₹{principal:,.0f}\n"
            f"   Interest Rate  : {annual_rate}% per annum\n"
            f"   Tenure         : {tenure_months:.0f} months ({tenure_months/12:.1f} years)\n"
            f"   Monthly EMI    : ₹{emi:,.0f}\n"
            f"   Total Payment  : ₹{total_payment:,.0f}\n"
            f"   Total Interest : ₹{total_interest:,.0f}"
        )

    # ── GENERAL MATH EXPRESSION ───────────────────────────────────────────
    # If the input is NOT an EMI request, treat it as a math expression.
    # Example: "sqrt(144) + 100" or "500000 * 0.01 * (1.01**36)"

    # Build a safe namespace of allowed math functions
    # We ONLY allow functions from the math module + abs + round
    # This prevents dangerous eval() usage like __import__('os').system('rm -rf /')
    _SAFE = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    _SAFE.update({"abs": abs, "round": round})

    # Security check: scan for any identifier (word) not in our safe list
    # re.findall(r"[a-zA-Z_]\w*", expression) finds all word-like tokens
    # Example: "sqrt(144)" → ["sqrt"]  ← safe
    # Example: "__import__('os')" → ["__import__", "os"]  ← unsafe!
    identifiers = re.findall(r"[a-zA-Z_]\w*", expression)
    unsafe = [i for i in identifiers if i not in _SAFE]
    if unsafe:
        return f"Error: unsafe identifiers detected: {unsafe}. Only math functions allowed."

    try:
        # eval() executes a string as Python code
        # {"__builtins__": {}} removes ALL built-in functions (no print, open, etc.)
        # _SAFE provides ONLY our approved math functions
        result = eval(expression, {"__builtins__": {}}, _SAFE)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {e}"


# ═══════════════════════════════════════════════════════════════════════════
# TOOL 3: WEB SEARCH
# ═══════════════════════════════════════════════════════════════════════════
@tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo for current banking information such as
    RBI repo rate, current market interest rates, recent RBI policy updates,
    or any general information not found in the bank's internal documents.
    Input: a search query string."""

    # DuckDuckGoSearchRun: LangChain wrapper around DuckDuckGo
    # Free, no API key, good for current/recent information
    # Returns a string summary of top search results
    return DuckDuckGoSearchRun().run(query)


# ─────────────────────────────────────────────────────────────────────────
# TOOLS LIST — passed to the agent
# ─────────────────────────────────────────────────────────────────────────
# With @tool decorator, just pass the function names directly (no Tool() wrapper)
tools = [document_search, loan_emi_calculator, web_search]

print("✅ All 3 tools defined using @tool decorator")
print(f"   Tool 1: {document_search.name}")
print(f"   Tool 2: {loan_emi_calculator.name}")
print(f"   Tool 3: {web_search.name}")

✅ All 3 tools defined using @tool decorator
   Tool 1: document_search
   Tool 2: loan_emi_calculator
   Tool 3: web_search


### 4a — Test each tool independently

In [9]:
pip install -U ddgs

  Attempting uninstall: ddgs
    Found existing installation: ddgs 9.11.3
    Uninstalling ddgs-9.11.3:
      Successfully uninstalled ddgs-9.11.3
Note: you may need to restart the kernel to use updated packages.


In [39]:
# ─────────────────────────────────────────────
# With @tool decorator, tools are already created as StructuredTool objects.
# No need to instantiate them — just use them directly.
# To test tools manually, use .invoke("input") instead of .run()
# ─────────────────────────────────────────────
tools = [document_search, loan_emi_calculator, web_search]

# Test 1: Document Search
print("=" * 60)
print("📄 TOOL TEST 1: Document Search")
print("=" * 60)
result = document_search.invoke("What is the minimum CIBIL score for personal loan?")
print(result[:500], "...\n")

# Test 2: EMI Calculator (named parameters)
print("=" * 60)
print("🔢 TOOL TEST 2: EMI Calculator (named parameters)")
print("=" * 60)
result = loan_emi_calculator.invoke("EMI: principal=500000, rate=12, tenure=36")
print(result, "\n")

# Test 3: EMI Calculator (home loan example)
print("=" * 60)
print("🏠 TOOL TEST 3: EMI Calculator (home loan)")
print("=" * 60)
result = loan_emi_calculator.invoke("EMI: principal=5000000, rate=8.5, tenure=240")
print(result, "\n")

# Test 4: Web Search
print("=" * 60)
print("🌐 TOOL TEST 4: Web Search")
print("=" * 60)
result = web_search.invoke("Iran and Israel war")
print(result[:400], "...")

📄 TOOL TEST 1: Document Search
[Source 1: customer_faq.txt]
Q: Can I get a loan if my CIBIL score is below 700?
A: Standard personal loans require a minimum CIBIL score of 700. However, secured loans
   (against FD, gold, or property) may be available with a lower credit score.
   We recommend improving your credit score before applying for unsecured loans.

SECTION C: CREDIT CARDS
------------------------

[Source 2: loan_policy.txt]
3. HOME LOAN ELIGIBILITY CRITERIA
-----------------------------------
- Minimum age: 23 year ...

🔢 TOOL TEST 2: EMI Calculator (named parameters)
💰 EMI Calculation:
   Loan Amount    : ₹500,000
   Interest Rate  : 12.0% per annum
   Tenure         : 36 months (3.0 years)
   Monthly EMI    : ₹16,607
   Total Payment  : ₹597,858
   Total Interest : ₹97,858 

🏠 TOOL TEST 3: EMI Calculator (home loan)
💰 EMI Calculation:
   Loan Amount    : ₹5,000,000
   Interest Rate  : 8.5% per annum
   Tenure         : 240 months (20.0 years)
   Monthly EMI    : ₹43,391
  

---
## 🧠 Phase 5 — The ReAct Agent (The Brain)

### 📖 What Is a ReAct Agent?
ReAct stands for **Reasoning + Acting**. It's a pattern where the agent:

1. **Thought** — Thinks about what it needs to do
2. **Action** — Picks a tool and calls it
3. **Observation** — Reads the tool's output
4. Repeats until it has enough info, then gives a **Final Answer**

### Example Agent Loop:
```
Question: What is the EMI for a ₹10 lakh personal loan at 14% for 2 years?

Thought: I need to calculate the EMI. I should use the loan_emi_calculator tool.
Action: loan_emi_calculator
Action Input: EMI: principal=1000000, rate=14, tenure=24
Observation: Monthly EMI: ₹48,013 | Total Payment: ₹11,52,312 | Total Interest: ₹1,52,312

Thought: I have the EMI. I can now give the final answer.
Final Answer: For a ₹10 lakh personal loan at 14% for 24 months, your monthly EMI will be ₹48,013...
```

### 🤖 Why Mistral-7B?
- Completely free, runs via HuggingFace Hub API
- Good at following ReAct reasoning format
- 7 billion parameters — powerful enough for complex banking questions
- No GPU needed on your end — HuggingFace runs it on their servers


In [14]:
pip install -U langchain langchain-openai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 15.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-openai]3/4 [langchain-openai]
Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.


In [40]:
!pip install -q langchain-groq

from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic import hub

# ─────────────────────────────────────────────
# STEP 1: Initialize LLM via Groq (free, fast)
# ─────────────────────────────────────────────
GROQ_API_KEY = ""  # ← paste from console.groq.com

llm = ChatGroq(
    model="llama-3.1-8b-instant",        # free, great for ReAct agents
    api_key=GROQ_API_KEY,
    temperature=0.1,
    max_tokens=512,
)
print("✅ LLM connected via Groq!")

# ─────────────────────────────────────────────
# STEP 2: Tools
# ─────────────────────────────────────────────
tools = [document_search, loan_emi_calculator, web_search]

print(f"\n🔧 Tools available:")
for t in tools:
    print(f"   - {t.name}: {t.description[:60]}...")

# ─────────────────────────────────────────────
# STEP 3: Load ReAct prompt
# ─────────────────────────────────────────────
print("\n⏳ Loading ReAct prompt...")
react_prompt = hub.pull("hwchase17/react")
print("✅ Prompt loaded!")

# ─────────────────────────────────────────────
# STEP 4 + 5: Create agent + executor
# ─────────────────────────────────────────────
agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=6,
)

print("\n✅ Banking Agent ready!")

✅ LLM connected via Groq!

🔧 Tools available:
   - document_search: Search the bank's internal documents for information about l...
   - loan_emi_calculator: Calculate loan EMI (Equated Monthly Installment) or evaluate...
   - web_search: Search the web using DuckDuckGo for current banking informat...

⏳ Loading ReAct prompt...
✅ Prompt loaded!

✅ Banking Agent ready!


---
## 🏃 Phase 6 — Run the Banking Agent

### 📖 What to Expect
With `verbose=True`, you'll see the agent's full thinking process:
- **Thought**: What the agent is thinking
- **Action**: Which tool it chose
- **Action Input**: What it sent to the tool
- **Observation**: What the tool returned
- **Final Answer**: The agent's final response to you


In [31]:
!pip show huggingface_hub langchain_huggingface

Name: huggingface_hub
Version: 1.6.0
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache-2.0
Location: /opt/anaconda3/lib/python3.13/site-packages
Requires: filelock, fsspec, hf-xet, httpx, packaging, pyyaml, tqdm, typer, typing-extensions
Required-by: langchain-huggingface, sentence-transformers, tokenizers, transformers
---
Name: langchain-huggingface
Version: 1.2.1
Summary: An integration package connecting Hugging Face and LangChain.
Home-page: https://docs.langchain.com/oss/python/integrations/providers/huggingface
Author: 
Author-email: 
License: MIT
Location: /opt/anaconda3/lib/python3.13/site-packages
Requires: huggingface-hub, langchain-core, tokenizers
Required-by: 


In [41]:
# ─────────────────────────────────────────────
# EXAMPLE 1: Loan Eligibility Question
# Agent should: use document_search tool
# ─────────────────────────────────────────────
print("\n" + "🏦 QUESTION 1: Loan Eligibility".center(60, "="))
result = agent_executor.invoke({
        "input": "I earn ₹35,000 per month and my CIBIL score is 720. Am I eligible for a personal loan?"})
print("\n" + "FINAL ANSWER ".center(60, "─"))
print(result["output"])


===============🏦 QUESTION 1: Loan Eligibility===============


> Entering new AgentExecutor chain...
Thought: To determine eligibility for a personal loan, I need to know the bank's personal loan eligibility criteria.
Action: document_search
Action Input: personal loan eligibility criteria[Source 1: loan_policy.txt]
1. PERSONAL LOAN ELIGIBILITY CRITERIA
--------------------------------------
- Minimum age: 21 years | Maximum age: 60 years at loan maturity
- Minimum monthly income: ₹25,000 (salaried) | ₹30,000 (self-employed)
- Minimum credit score (CIBIL): 700 and above
- Minimum employment tenure: 2 years (salaried) | 3 years (self-employed)
- Indian resident or NRI with valid documentation

[Source 2: loan_policy.txt]
3. HOME LOAN ELIGIBILITY CRITERIA
-----------------------------------
- Minimum age: 23 years | Maximum age: 70 years at loan maturity
- Minimum monthly income: ₹40,000
- Minimum credit score (CIBIL): 750 and above
- Maximum LTV (Loan-to-Value): 80% of property value



In [25]:
# ─────────────────────────────────────────────
# EXAMPLE 2: EMI Calculation
# Agent should: use loan_emi_calculator tool
# ─────────────────────────────────────────────
print("\n" + "🔢 QUESTION 2: EMI Calculation".center(60, "="))
result = agent_executor.invoke({
    "input": "I want to take a home loan of ₹30 lakhs at 9% interest for 20 years. What will be my monthly EMI and total interest paid?"
})
print("\n" + "FINAL ANSWER ".center(60, "─"))
print(result["output"])



===============🔢 QUESTION 2: EMI Calculation================


> Entering new AgentExecutor chain...
Thought: To find the monthly EMI and total interest paid, I need to calculate the EMI using the given loan amount, interest rate, and tenure.
Action: loan_emi_calculator
Action Input: 'EMI: principal=30000000, rate=9, tenure=240'💰 EMI Calculation:
   Loan Amount    : ₹30,000,000
   Interest Rate  : 9.0% per annum
   Tenure         : 240 months (20.0 years)
   Monthly EMI    : ₹269,918
   Total Payment  : ₹64,780,269
   Total Interest : ₹34,780,269Question: I want to take a home loan of ₹30 lakhs at 9% interest for 20 years. What will be my monthly EMI and total interest paid?
Thought: To find the monthly EMI and total interest paid, I need to calculate the EMI using the given loan amount, interest rate, and tenure.
Action: loan_emi_calculator
Action Input: 'EMI: principal=30000000, rate=9, tenure=240'💰 EMI Calculation:
   Loan Amount    : ₹30,000,000
   Interest Rate  : 9.0% per annum


In [26]:
# ─────────────────────────────────────────────
# EXAMPLE 3: Credit Card FAQ
# Agent should: use document_search tool
# ─────────────────────────────────────────────
print("\n" + "💳 QUESTION 3: Credit Card Question".center(60, "="))
result = agent_executor.invoke({
    "input": "What credit card can I get if my annual income is ₹8 lakhs? What rewards will I earn?"
})
print("\n" + "FINAL ANSWER ".center(60, "─"))
print(result["output"])



=============💳 QUESTION 3: Credit Card Question=============


> Entering new AgentExecutor chain...
Thought: To answer this question, I need to find information about credit cards and their eligibility criteria.
Action: document_search
Action Input: credit card policies and eligibility criteria[Source 1: customer_faq.txt]
SECTION C: CREDIT CARDS
------------------------

Q: How do I apply for a credit card?
A: You can apply online at our website, at any branch, or through our mobile app.
   Eligibility: minimum age 21 years, minimum annual income ₹3,00,000, CIBIL score 700+.

[Source 2: credit_card_policy.txt]
4. CREDIT LIMIT ENHANCEMENT
-----------------------------
  - Request after 6 months of card usage
  - Good repayment history and income proof required
  - Enhancement of up to 30% can be done instantly online
  - Higher enhancements require income document re-verification

[Source 3: credit_card_policy.txt]
NATIONAL BANK — CREDIT CARD POLICY & FEATURES

1. CREDIT CARD TYPES & 

In [27]:
# ─────────────────────────────────────────────
# EXAMPLE 4: Multi-step question (uses multiple tools)
# Agent might: search docs + calculate
# ─────────────────────────────────────────────
print("\n" + "🧠 QUESTION 4: Multi-step Banking Query".center(60, "="))
result = agent_executor.invoke({
    "input": "I want a personal loan of ₹8 lakhs. What interest rate will I get, and what will my EMI be for a 4-year tenure?"
})
print("\n" + "FINAL ANSWER ".center(60, "─"))
print(result["output"])



===========🧠 QUESTION 4: Multi-step Banking Query===========


> Entering new AgentExecutor chain...
Thought: To find the interest rate and EMI, I need to search the bank's internal documents for information about loan policies and interest rates.
Action: document_search
Action Input: loan policies and interest rates[Source 1: loan_policy.txt]
NATIONAL BANK — PERSONAL LOAN & HOME LOAN POLICY DOCUMENT

[Source 2: customer_faq.txt]
Q: What is the interest rate on savings accounts?
A: Regular Savings Account: 3.5% per annum. Premium Savings Account: 4.0% per annum.
   Interest is calculated daily and credited quarterly.

Q: How many free ATM transactions do I get per month?
A: You get 5 free transactions at our own bank ATMs (unlimited at home branch).
   For other bank ATMs, you get 3 free transactions per month. After that,
   ₹20 per transaction is charged.

[Source 3: loan_policy.txt]
2. PERSONAL LOAN DETAILS
--------------------------
- Loan amount: ₹50,000 to ₹40,00,000
- Interest 

In [28]:
# ─────────────────────────────────────────────
# EXAMPLE 5: Web search question
# Agent should: use web_search tool
# ─────────────────────────────────────────────
print("\n" + "🌐 QUESTION 5: Current RBI Information".center(60, "="))
result = agent_executor.invoke({
    "input": "What is the current RBI repo rate and how does it affect home loan interest rates?"
})
print("\n" + "FINAL ANSWER ".center(60, "─"))
print(result["output"])



===========🌐 QUESTION 5: Current RBI Information============


> Entering new AgentExecutor chain...
Thought: To answer this question, I need to find information about the current RBI repo rate and its impact on home loan interest rates.
Action: web_search
Action Input: "current RBI repo rate and impact on home loan interest rates"heReserveBankofIndia(RBI) has decided to keep thereporateunchanged at 5.25% in its February 2026 monetary policy review, bringing much-needed stability forhomeloanborrowers across the country. The benchmarkinterestrateinIndiawas last recorded at 5.25 percent. This page provides -IndiaInterestRate- actual values, historical data, forecast, chart, statistics, economic calendar and news. Reporateis theinterestrateat which theRBIlends money to commercialbanks.ICRA onImpactonprofitability ofbanksbecause ofreporatecut. We expect NIMs as % of the advances will contract by 15 bps for thebanks, which will lead to a decline of 0.80% in ROE forbanks. Homeloaninterestra

---
## 💬 Phase 7 — Interactive Banking Chat

### 📖 How to Use
- Type any banking question and press Enter
- The agent will reason through it and respond
- Type `quit` to exit the chat loop

### 💡 Try These Questions
- "Can I get a gold credit card with ₹7 lakh annual salary?"
- "What happens if I miss 2 EMI payments?"
- "Calculate EMI for ₹15 lakh loan at 11.5% for 5 years"
- "What documents do I need for a home loan?"
- "Is my deposit safe if the bank goes bankrupt?"


In [29]:
from IPython.display import display, Markdown

print("🏦 National Bank AI Assistant")
print("=" * 50)
print("Ask me anything about loans, credit cards, FDs, or banking policies.")
print("Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "q", "bye"):
        print("Thank you for banking with us! Goodbye 👋")
        break

    print("\n⏳ Agent is thinking...\n")

    try:
        result = agent_executor.invoke({"input": user_input})
        answer = result.get("output", "Sorry, I could not generate an answer.")
    except Exception as e:
        answer = f"An error occurred: {e}"

    display(Markdown(f"**🤖 Assistant:** {answer}"))
    print("\n" + "-" * 50 + "\n")


🏦 National Bank AI Assistant
Ask me anything about loans, credit cards, FDs, or banking policies.
Type 'quit' to exit.



You:  i want apply for 200,000 loan. my current salary is 130000



⏳ Agent is thinking...



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to know the eligibility criteria for a loan and the required documents.
Action: document_search
Action Input: loan eligibility criteria and required documents[Source 1: loan_policy.txt]
1. PERSONAL LOAN ELIGIBILITY CRITERIA
--------------------------------------
- Minimum age: 21 years | Maximum age: 60 years at loan maturity
- Minimum monthly income: ₹25,000 (salaried) | ₹30,000 (self-employed)
- Minimum credit score (CIBIL): 700 and above
- Minimum employment tenure: 2 years (salaried) | 3 years (self-employed)
- Indian resident or NRI with valid documentation

[Source 2: loan_policy.txt]
3. HOME LOAN ELIGIBILITY CRITERIA
-----------------------------------
- Minimum age: 23 years | Maximum age: 70 years at loan maturity
- Minimum monthly income: ₹40,000
- Minimum credit score (CIBIL): 750 and above
- Maximum LTV (Loan-to-Value): 80% of property value

[Source 3: loan_policy.txt]

**🤖 Assistant:** Agent stopped due to iteration limit or time limit.


--------------------------------------------------



You:  quit


Thank you for banking with us! Goodbye 👋


---
## 📊 Phase 8 — Evaluation

This section measure these — which is rare and impressive in portfolios.

### Metrics We'll Measure
| Metric | What It Measures |
|--------|-----------------|
| **Keyword Hit Rate** | Does the retrieved context contain expected keywords? |
| **Retrieval Latency** | How many seconds does retrieval take? |
| **Similarity Score** | How confident is FAISS about the top result? |


In [46]:
import time
import pandas as pd
from IPython.display import display

# ─────────────────────────────────────────────
# Ground Truth Evaluation Set
# Each item has: a question, and keywords we EXPECT to find in the retrieved context
# ─────────────────────────────────────────────
eval_set = [
    {
        "question": "What is the minimum CIBIL score for personal loan?",
        "expected_keywords": ["700", "cibil", "credit score", "personal loan"],
        "category": "Loan Eligibility"
    },
    {
        "question": "What are the interest charges on credit card outstanding?",
        "expected_keywords": ["3.5%", "interest", "month", "outstanding"],
        "category": "Credit Card"
    },
    {
        "question": "How do I block a lost debit card?",
        "expected_keywords": ["block", "helpline", "mobile app", "lost"],
        "category": "Customer FAQ"
    },
    {
        "question": "What documents are needed for home loan?",
        "expected_keywords": ["pan", "aadhaar", "salary", "bank statement"],
        "category": "Loan Documents"
    },
    {
        "question": "What is the annual fee for Gold credit card?",
        "expected_keywords": ["1,000", "gold", "annual fee", "waived"],
        "category": "Credit Card"
    },
    {
        "question": "What is the FD interest rate for 2 years?",
        "expected_keywords": ["7.50", "7.25", "fixed deposit", "year"],
        "category": "Fixed Deposit"
    },
]

def evaluate_retrieval(rag_pipeline, eval_data):
    """
    Evaluate the RAG pipeline's retrieval quality.

    For each question:
    1. Retrieve top-k chunks
    2. Combine all chunk text into one string
    3. Check how many expected keywords appear in that combined text
    4. Record latency and similarity score
    """
    records = []

    for item in eval_data:
        # Time the retrieval
        start_time = time.time()
        results = rag_pipeline.retrieve(item["question"])
        latency = round(time.time() - start_time, 3)

        # Combine all retrieved chunks into one text block
        combined_context = " ".join(r["content"].lower() for r in results)

        # Count how many expected keywords appear in the context
        keywords = item["expected_keywords"]
        hits = sum(1 for kw in keywords if kw.lower() in combined_context)
        hit_rate = round(hits / len(keywords) * 100, 1)

        # Get the similarity score of the best result
        top_score = results[0]["score"] if results else None
        top_source = Path(results[0]["metadata"].get("source", "?")).name if results else "N/A"

        records.append({
            "Category": item["category"],
            "Question": item["question"][:50] + "...",
            "Keywords Hit": f"{hits}/{len(keywords)} ({hit_rate}%)",
            "Top Source": top_source,
            "Similarity Score": f"{top_score:.4f}" if top_score else "N/A",
            "Latency (s)": latency,
        })

    return pd.DataFrame(records)

# Run evaluation
print("⏳ Running retrieval evaluation...")
eval_df = evaluate_retrieval(rag, eval_set)

print("\n📊 RETRIEVAL EVALUATION RESULTS")
print("=" * 70)
display(eval_df)

# Summary stats
hit_rates = [float(r.split("(")[1].replace("%)","")) for r in eval_df["Keywords Hit"]]
avg_latency = eval_df["Latency (s)"].mean()

print(f"\n📈 SUMMARY:")
print(f"   Average Keyword Hit Rate : {sum(hit_rates)/len(hit_rates):.1f}%")
print(f"   Average Retrieval Latency: {avg_latency:.3f} seconds")
print(f"   Questions Evaluated      : {len(eval_set)}")


⏳ Running retrieval evaluation...

📊 RETRIEVAL EVALUATION RESULTS


,Category,Question,Keywords Hit,Top Source,Similarity Score,Latency (s)
0,Loan Eligibility,What is the minimum CIBIL score for personal l...,4/4 (100.0%),customer_faq.txt,0.4528,0.313
1,Credit Card,What are the interest charges on credit card o...,4/4 (100.0%),customer_faq.txt,0.7249,0.011
2,Customer FAQ,How do I block a lost debit card?...,4/4 (100.0%),customer_faq.txt,0.5456,0.010
3,Loan Documents,What documents are needed for home loan?...,4/4 (100.0%),loan_policy.txt,1.0072,0.008
4,Credit Card,What is the annual fee for Gold credit card?...,4/4 (100.0%),credit_card_policy.txt,0.6910,0.010
5,Fixed Deposit,What is the FD interest rate for 2 years?...,3/4 (75.0%),customer_faq.txt,0.5492,0.009



📈 SUMMARY:
   Average Keyword Hit Rate : 95.8%
   Average Retrieval Latency: 0.060 seconds
   Questions Evaluated      : 6


---
## 💾 Phase 9 — Save & Reload Index

### 📖 Why Save the Index?
Embedding documents takes time (1-2 minutes for small docs, longer for large ones).
Saving the FAISS index means you only embed once — future runs load in seconds.


In [43]:
# Save the index to disk
rag.save_index()

# Demonstrate reloading works
print("\n⏳ Testing index reload...")
rag_reloaded = BankingRAGPipeline()
rag_reloaded.load_index()

# Verify it works correctly
test_result = rag_reloaded.retrieve("What is the minimum balance for savings account?")
print("\n✅ Reload successful! Test retrieval:")
print(f"   Top result preview: {test_result[0]['content'][:150]}...")


💾 Index saved to 'faiss_index/' folder

⏳ Testing index reload...
⏳ Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
   (This downloads ~90MB on first run, then cached locally)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BankingRAGPipeline initialized!
   Chunk size: 500 | Overlap: 50 | Top-K: 6
📂 Index loaded from 'faiss_index/'

✅ Reload successful! Test retrieval:
   Top result preview: NATIONAL BANK — CUSTOMER FREQUENTLY ASKED QUESTIONS (FAQ)

SECTION A: SAVINGS ACCOUNT
-----...


---
## 🖥️ Phase 10 — Generate Streamlit Web App

### 📖 What This Does
Generates a complete `banking_app.py` file you can run as a web application.
This is what you'd demo to employers or users.

### How to Run
```bash
# In terminal:
streamlit run banking_app.py

# In Colab (with public URL):
# Run the cell below which uses localtunnel
```
 Below code is base version.

banking_app_code = '''
import os, tempfile, math, re, streamlit as st
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import Tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

st.set_page_config(page_title="🏦 Banking AI Assistant", page_icon="🏦", layout="wide")

st.title("🏦 National Bank AI Assistant")
st.caption("Powered by Mistral-7B | RAG + Agentic AI | Open Source Stack")

# ── Sidebar ──
with st.sidebar:
    st.header("⚙️ Setup")
    hf_token = st.text_input("HuggingFace Token", type="password",
                              help="Get free token at huggingface.co/settings/tokens")
    model = st.selectbox("LLM Model", [
        "mistralai/Mistral-7B-Instruct-v0.2",
        "HuggingFaceH4/zephyr-7b-beta"
    ])
    st.divider()
    st.markdown("### 📄 Upload Banking Documents")
    uploaded = st.file_uploader("PDF or TXT files", type=["pdf","txt","md"],
                                 accept_multiple_files=True)
    if st.button("📥 Ingest Documents", use_container_width=True):
        if not uploaded:
            st.warning("Please upload files first.")
        else:
            with st.spinner("Processing documents..."):
                embeddings = HuggingFaceEmbeddings(
                    model_name="sentence-transformers/all-MiniLM-L6-v2",
                    model_kwargs={"device": "cpu"},
                    encode_kwargs={"normalize_embeddings": True}
                )
                splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                all_chunks = []
                with tempfile.TemporaryDirectory() as d:
                    for f in uploaded:
                        p = Path(d) / f.name
                        p.write_bytes(f.read())
                        try:
                            docs = PyPDFLoader(str(p)).load() if f.name.endswith(".pdf") else TextLoader(str(p)).load()
                            all_chunks.extend(splitter.split_documents(docs))
                        except Exception as e:
                            st.warning(f"Could not load {f.name}: {e}")
                if all_chunks:
                    st.session_state.vs = FAISS.from_documents(all_chunks, embeddings)
                    st.session_state.chunk_count = len(all_chunks)
            st.success(f"✅ Indexed {st.session_state.get('chunk_count', 0)} chunks!")

    st.divider()
    if "chunk_count" in st.session_state:
        st.metric("Chunks Indexed", st.session_state.chunk_count)
        st.success("Knowledge base ready ✅")
    else:
        st.info("Upload documents to get started")

# ── Sample Questions ──
st.markdown("### 💡 Try asking:")
cols = st.columns(2)
sample_questions = [
    "Am I eligible for a personal loan with ₹30k salary and 720 CIBIL?",
    "Calculate EMI for ₹10 lakh at 12% for 3 years",
    "What documents are needed for a home loan?",
    "What credit card suits an ₹8 lakh annual income?",
]
for i, q in enumerate(sample_questions):
    if cols[i % 2].button(q, use_container_width=True):
        st.session_state.prefill = q

# ── Chat History ──
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── Chat Input ──
prefill = st.session_state.pop("prefill", "")
if question := st.chat_input("Ask about loans, EMI, credit cards, policies...") or prefill:
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    if not hf_token:
        answer = "⚠️ Please enter your HuggingFace API token in the sidebar to use the assistant."
    else:
        os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

        def doc_search(q):
            if "vs" not in st.session_state:
                return "No documents uploaded yet. Please upload banking documents in the sidebar."
            results = st.session_state.vs.similarity_search(q, k=4)
            return "\n\n".join(f"[{i+1}] {r.page_content}" for i, r in enumerate(results))

        _safe = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        _safe.update({"abs": abs, "round": round})

        def emi_calc(expr):
            emi_match = re.search(r"(?i)emi.*principal[=:]\s*([\d.]+).*rate[=:]\s*([\d.]+).*tenure[=:]\s*([\d.]+)", expr)
            if emi_match:
                p, r_ann, n = float(emi_match.group(1)), float(emi_match.group(2)), float(emi_match.group(3))
                r = r_ann / 12 / 100
                emi = p * r * (1+r)**n / ((1+r)**n - 1) if r else p/n
                return f"EMI: ₹{emi:,.0f}/month | Total: ₹{emi*n:,.0f} | Interest: ₹{emi*n-p:,.0f}"
            try:
                return str(eval(expr, {"__builtins__": {}}, _safe))
            except Exception as e:
                return str(e)

        tools = [
            Tool("document_search", doc_search, "Search bank documents for policies, eligibility, FAQs."),
            Tool("loan_emi_calculator", emi_calc, "Calculate loan EMI. Format: EMI: principal=X, rate=Y, tenure=Z"),
            Tool("web_search", DuckDuckGoSearchRun().run, "Search web for current rates or banking news."),
        ]
        llm = HuggingFaceEndpoint(repo_id=model, huggingfacehub_api_token=hf_token,
                                   temperature=0.1, max_new_tokens=512)
        agent_exec = AgentExecutor(
            agent=create_react_agent(llm, tools, hub.pull("hwchase17/react")),
            tools=tools, handle_parsing_errors=True, max_iterations=6, verbose=False
        )
        with st.chat_message("assistant"):
            with st.spinner("🤖 Thinking..."):
                try:
                    answer = agent_exec.invoke({"input": question})["output"]
                except Exception as e:
                    answer = f"Error: {e}"
            st.markdown(answer),

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("banking_app.py", "w") as f:
    f.write(banking_app_code)

print("✅ banking_app.py generated!")
print("\nTo run the web app:")
print("  Local  : streamlit run banking_app.py")
print("  Colab  : Run the next cell")


## Banking_APP

import os, math, re, tempfile
import streamlit as st
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic import hub
from langchain_groq import ChatGroq

st.set_page_config(page_title="🏦 Banking AI Assistant", page_icon="🏦", layout="wide")
st.title("🏦 National Bank AI Assistant")
st.caption("Powered by LLaMA-3.1 | RAG + Agentic AI | Open Source Stack")

# ── Sidebar ──
with st.sidebar:
    st.header("⚙️ Setup")
    groq_api_key = st.text_input("Groq API Key", type="password",
                                  help="Get free key at console.groq.com")
    st.divider()
    st.markdown("### 📄 Upload Banking Documents")
    uploaded = st.file_uploader("PDF or TXT files", type=["pdf", "txt", "md"],
                                 accept_multiple_files=True)

    if st.button("📥 Ingest Documents", use_container_width=True):
        if not uploaded:
            st.warning("Please upload files first.")
        else:
            with st.spinner("Processing documents..."):
                embeddings = HuggingFaceEmbeddings(
                    model_name="sentence-transformers/all-MiniLM-L6-v2",
                    model_kwargs={"device": "cpu"},
                    encode_kwargs={"normalize_embeddings": True}
                )
                splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                all_chunks = []
                with tempfile.TemporaryDirectory() as d:
                    for f in uploaded:
                        p = Path(d) / f.name
                        p.write_bytes(f.read())
                        try:
                            if f.name.endswith(".pdf"):
                                docs = PyPDFLoader(str(p)).load()
                            else:
                                docs = TextLoader(str(p), encoding="utf-8").load()
                            all_chunks.extend(splitter.split_documents(docs))
                        except Exception as e:
                            st.warning(f"Could not load {f.name}: {e}")
                if all_chunks:
                    st.session_state.vs = FAISS.from_documents(all_chunks, embeddings)
                    st.session_state.chunk_count = len(all_chunks)
            st.success(f"✅ Indexed {st.session_state.get('chunk_count', 0)} chunks!")

    st.divider()
    if "chunk_count" in st.session_state:
        st.metric("Chunks Indexed", st.session_state.chunk_count)
        st.success("Knowledge base ready ✅")
    else:
        st.info("Upload documents to get started")

# ── Sample Questions ──
st.markdown("### 💡 Try asking:")
cols = st.columns(2)
sample_questions = [
    "Am I eligible for a personal loan with ₹30k salary and 720 CIBIL?",
    "Calculate EMI for ₹10 lakh at 12% for 3 years",
    "What documents are needed for a home loan?",
    "What credit card suits an ₹8 lakh annual income?",
]
for i, q in enumerate(sample_questions):
    if cols[i % 2].button(q, use_container_width=True):
        st.session_state.prefill = q

# ── Chat History ──
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── Chat Input ──
prefill = st.session_state.pop("prefill", "")
question = st.chat_input("Ask about loans, EMI, credit cards, policies...") or prefill

if question:
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    if not groq_api_key:
        answer = "⚠️ Please enter your Groq API key in the sidebar."
    else:
        # ── Tools ──
        def doc_search(q):
            if "vs" not in st.session_state:
                return "No documents uploaded yet. Please upload banking documents in the sidebar."
            results = st.session_state.vs.similarity_search(q, k=6)
            return "\n\n".join(f"[{i+1}] {r.page_content}" for i, r in enumerate(results))

        _safe = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        _safe.update({"abs": abs, "round": round})

        def emi_calc(expr):
            emi_match = re.search(
                r"(?i)emi.*principal[=:]\s*([\d.]+).*rate[=:]\s*([\d.]+).*tenure[=:]\s*([\d.]+)",
                expr
            )
            if emi_match:
                p = float(emi_match.group(1))
                r_ann = float(emi_match.group(2))
                n = float(emi_match.group(3))
                r = r_ann / 12 / 100
                emi = p * r * (1+r)**n / ((1+r)**n - 1) if r else p/n
                return f"EMI: ₹{emi:,.0f}/month | Total: ₹{emi*n:,.0f} | Interest: ₹{emi*n-p:,.0f}"
            try:
                return str(eval(expr, {"__builtins__": {}}, _safe))
            except Exception as e:
                return str(e)

        tools = [
            tool("document_search", doc_search,
                 "Search bank documents for policies, eligibility, FAQs."),
            tool("loan_emi_calculator", emi_calc,
                 "Calculate loan EMI. Format: EMI: principal=X, rate=Y, tenure=Z"),
            tool("web_search", DuckDuckGoSearchRun().run,
                 "Search web for current rates or banking news."),
        ]

        llm = ChatGroq(
            model="llama-3.1-8b-instant",
            api_key=groq_api_key,
            temperature=0.1,
            max_tokens=512,
        )

        react_prompt = hub.pull("hwchase17/react")
        agent_exec = AgentExecutor(
            agent=create_react_agent(llm, tools, react_prompt),
            tools=tools,
            handle_parsing_errors=True,
            max_iterations=6,
            verbose=False,
        )

        with st.chat_message("assistant"):
            with st.spinner("🤖 Thinking..."):
                try:
                    answer = agent_exec.invoke({"input": question})["output"]
                except Exception as e:
                    answer = f"Error: {e}"
            st.markdown(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})


In [45]:
# ── Launch in Colab via localtunnel ──
# Skip this cell if running locally — just use: streamlit run banking_app.py

!pip install -q localtunnel
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run", "banking_app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)
print("⏳ Starting tunnel... (a URL will appear below)")
!npx localtunnel --port 8501


ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel
⏳ Starting tunnel... (a URL will appear below)
zsh:1: command not found: npx
